In [0]:

from __future__ import annotations

import argparse
import json

from inference_artifacts import (
    build_spark_session,
    build_stage06_compatible_horizon_feature_dataframes,
    get_base_path,
    get_step_dir,
    read_step_artifact,
    resolve_inference_run_context,
    write_step_artifact,
)


def run_feature_rows_step(run_id: str, run_ts: str) -> tuple[dict, str]:
    base_path = get_base_path()

    live_payload = read_step_artifact(
        base_path=base_path,
        run_id=run_id,
        step_name="live_station",
        filename="live_station.json",
    )
    weather_payload = read_step_artifact(
        base_path=base_path,
        run_id=run_id,
        step_name="weather",
        filename="weather.json",
    )
    synthetic_history_payload = read_step_artifact(
        base_path=base_path,
        run_id=run_id,
        step_name="synthetic_history",
        filename="synthetic_history.json",
    )

    station_id = str(live_payload["live_station"]["station_id"])
    canonical_station_id = str(live_payload["canonical_station_id"])

    spark = build_spark_session()
    try:
        feature_df, model_input_df, metadata = build_stage06_compatible_horizon_feature_dataframes(
            spark=spark,
            base_path=base_path,
            canonical_station_id=canonical_station_id,
            weather_rows=weather_payload["rows"],
            synthetic_history_rows=synthetic_history_payload["rows"],
        )

        if metadata["missing_gold_columns"]:
            raise ValueError(
                "Milestone 5 feature build is missing Stage-06 columns: "
                f"{metadata['missing_gold_columns']}"
            )
        if metadata["missing_model_input_columns"]:
            raise ValueError(
                "Milestone 5 feature build is missing model input columns: "
                f"{metadata['missing_model_input_columns']}"
            )

        step_dir = get_step_dir(base_path, run_id, "feature_rows")
        step_dir.mkdir(parents=True, exist_ok=True)

        stage06_compatible_path = str(step_dir / "stage06_compatible")
        model_input_path = str(step_dir / "model_input")

        feature_df.write.mode("overwrite").parquet(stage06_compatible_path)
        model_input_df.write.mode("overwrite").parquet(model_input_path)
    finally:
        spark.stop()

    payload = {
        "run_id": run_id,
        "run_ts": run_ts,
        "request_timestamp": live_payload["request_timestamp"],
        "station_id": station_id,
        "canonical_station_id": canonical_station_id,
        "feature_rows_path": stage06_compatible_path,
        "model_input_path": model_input_path,
        "row_count": metadata["row_count"],
        "horizon_start": metadata["horizon_start"],
        "horizon_end_exclusive": metadata["horizon_end_exclusive"],
        "gold_columns": metadata["gold_columns"],
        "model_column_groups": metadata["model_column_groups"],
        "missing_gold_columns": metadata["missing_gold_columns"],
        "missing_model_input_columns": metadata["missing_model_input_columns"],
    }

    artifact_path = write_step_artifact(
        base_path=base_path,
        run_id=run_id,
        step_name="feature_rows",
        payload=payload,
        filename="feature_rows.json",
    )
    return payload, artifact_path


def main() -> None:
    parser = argparse.ArgumentParser(description="Inference Step 04: stage-06-compatible feature rows")
    parser.add_argument("--run-id", default=None)
    parser.add_argument("--run-ts", default=None)
    args = parser.parse_args()

    run_id, run_ts = resolve_inference_run_context(args.run_id, args.run_ts)
    payload, artifact_path = run_feature_rows_step(run_id=run_id, run_ts=run_ts)

    print(
        json.dumps(
            {
                "step": "feature_rows",
                "run_id": run_id,
                "run_ts": run_ts,
                "artifact_path": artifact_path,
                "feature_rows_path": payload["feature_rows_path"],
                "model_input_path": payload["model_input_path"],
                "row_count": payload["row_count"],
            },
            indent=2,
        )
    )


In [0]:
import os
def store_databricks_params_as_env():
    if os.getenv("DATABRICKS_RUNTIME_VERSION"):
        # 1. Retrieve all parameters (widgets) as a dictionary
        # This returns a dict of { "parameter_name": "parameter_value" }
        all_params = dbutils.widgets.getAll()

        # 2. Store them in os.environ
        for key, value in all_params.items():
            os.environ[key] = str(value)

        print(f"Stored {len(all_params)} parameters in environment variables.")

def build_default_run_id() -> str:
    env_run_id = os.environ.get("INFERENCE_RUN_ID")
    if env_run_id and env_run_id.strip():
        return env_run_id.strip()

    pipeline_run_id = os.environ.get("PIPELINE_RUN_ID")
    if pipeline_run_id and pipeline_run_id.strip():
        return pipeline_run_id.strip()

    job_run_id = os.environ.get("PIPELINE_JOB_RUN_ID")
    if job_run_id and job_run_id.strip():
        repair_count = os.environ.get("PIPELINE_REPAIR_COUNT", "0").strip() or "0"
        return f"job_{job_run_id.strip()}_repair_{repair_count}"

    now_utc = datetime.datetime.now(timezone.utc)
    return f"run_{now_utc.strftime('%Y%m%d%H%M%S')}_{uuid.uuid4().hex[:8]}"


def build_default_run_ts() -> str:
    env_run_ts = os.environ.get("INFERENCE_RUN_TS")
    if env_run_ts and env_run_ts.strip():
        return env_run_ts.strip()

    pipeline_run_ts = os.environ.get("PIPELINE_RUN_TS")
    if pipeline_run_ts and pipeline_run_ts.strip():
        return pipeline_run_ts.strip()

    return datetime.datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")

def is_job():
    try:
        # Access the internal notebook context
        context_json = dbutils.notebook.entry_point.getDbutils().notebook().getContext().toJson()
        context = json.loads(context_json)
        
        # Check for the existence of jobId in tags
        return "jobId" in context.get("tags", {})
    except:
        return False

import datetime
from datetime import timedelta, timezone
import uuid

if is_job():
    print("Running as a Databricks Job")
else:
    print("Running interactively (Manual)")
    now_utc = datetime.datetime.now(timezone.utc)
    dbutils.widgets.text("INFERENCE_REQUEST_JSON", '{"name":"du Mont-Royal / Clark","lat":45.51941,"lon":-73.58685,"station_id":"218"}')
    dbutils.widgets.text("INFERENCE_RUN_ID", "20260420080036_e9483314") #f"{now_utc.strftime('%Y%m%d%H%M%S')}_{uuid.uuid4().hex[:8]}")
    dbutils.widgets.text("INFERENCE_HORIZON_STEPS", "3")

In [0]:
store_databricks_params_as_env()

request_json = os.getenv("INFERENCE_REQUEST_JSON")
run_id = os.getenv("INFERENCE_RUN_ID")
horizon_steps = os.getenv("INFERENCE_HORIZON_STEPS")

print("args request json:", request_json)
print("args horizon_steps id:", horizon_steps)
print("args run_id:", run_id)

run_id, run_ts = resolve_inference_run_context(run_id, build_default_run_ts())


In [0]:
# run_id, run_ts = resolve_inference_run_context(args.run_id, args.run_ts)
payload, artifact_path = run_feature_rows_step(run_id=run_id, run_ts=run_ts)

print(
    json.dumps(
        {
            "step": "feature_rows",
            "run_id": run_id,
            "run_ts": run_ts,
            "artifact_path": artifact_path,
            "feature_rows_path": payload["feature_rows_path"],
            "model_input_path": payload["model_input_path"],
            "row_count": payload["row_count"],
        },
        indent=2,
    )
)
